<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/02%20-%20Supervised%20Learning/solutions/02_supervised_solutions.ipynb)

# 02 — 监督学习：课堂练习参考答案（Solutions）

本 Notebook 汇总了 `02a`、`02b`、`02c` 三个模块中所有课堂练习（`02a-1` 至 `02c-3`）的完整可运行参考代码与结果解析。建议先在原讲义中独立动手尝试，再对照本文件核对思路。

---

## 0. 公共环境与 ESOL 数据准备

先把数据、10 个 RDKit 物理化学描述符和训练/测试集划分一次性准备好，方便后续各题直接调用。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "02 - Supervised Learning/solutions"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    RandomForestClassifier, GradientBoostingClassifier,
)
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

DATA_PATHS = [
    Path("../../01 - Basics/data/esol.csv"),
    Path("../01 - Basics/data/esol.csv"),
    Path("data/esol.csv"),
]
for p in DATA_PATHS:
    if p.exists():
        df = pd.read_csv(p)
        break

df = df.rename(columns={
    "Compound ID": "name",
    "measured log solubility in mols per litre": "logS",
})[["name", "smiles", "logS"]].copy()
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
df = df.dropna(subset=["mol"]).reset_index(drop=True)

DESC_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
    "NumHDonors": Descriptors.NumHDonors,
    "NumHAcceptors": Descriptors.NumHAcceptors,
    "NumRotatableBonds": Descriptors.NumRotatableBonds,
    "RingCount": Descriptors.RingCount,
    "NumAromaticRings": Descriptors.NumAromaticRings,
    "HeavyAtomCount": Descriptors.HeavyAtomCount,
    "FractionCSP3": Descriptors.FractionCSP3,
}
X = pd.DataFrame([{k: fn(m) for k, fn in DESC_FUNCS.items()} for m in df["mol"]])
y_reg = df["logS"].values

print(f"Loaded {len(df)} molecules, X shape = {X.shape}")

---

# 02a — 回归入门（Regression）练习答案

先运行 `02a` 中的数据划分、评估函数与标准化准备：

In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE
)
X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y_reg[idx_train], y_reg[idx_test]

def evaluate(model, X_tr, y_tr, X_te, y_te, label="model"):
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    return {
        "model": label,
        "Train RMSE": np.sqrt(mean_squared_error(y_tr, pred_tr)),
        "Test RMSE":  np.sqrt(mean_squared_error(y_te, pred_te)),
        "Test MAE":   mean_absolute_error(y_te, pred_te),
        "Test R2":    r2_score(y_te, pred_te),
    }

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
results = {}

### 练习 02a-1：实例化 k 近邻回归器并在原始特征上训练

**题目回顾**：实例化一个设置 `n_neighbors=10` 的 `KNeighborsRegressor`，赋值给变量 `knn_reg`，并用原始特征 `X_train`、`X_test` 调用 `evaluate` 查看其表现。

In [ ]:
knn_reg = KNeighborsRegressor(n_neighbors=10)
knn_reg.fit(X_train, y_train)
results["kNN (k=10, raw)"] = evaluate(knn_reg, X_train, y_train, X_test, y_test, "kNN (k=10, raw)")
pd.DataFrame([results["kNN (k=10, raw)"]]).set_index("model").round(3)

### 练习 02a-2：在标准化特征上训练支持向量回归（SVR）

**题目回顾**：训练一个默认参数的 `SVR()`（使用标准化后的 `X_train_s` 和 `X_test_s`），把评估结果存进 `results["SVR"]`，并对比它与标准化后 kNN 的表现。

In [ ]:
knn_s = KNeighborsRegressor(n_neighbors=10).fit(X_train_s, y_train)
results["kNN (k=10, scaled)"] = evaluate(knn_s, X_train_s, y_train, X_test_s, y_test, "kNN (k=10, scaled)")

svr_model = SVR()
svr_model.fit(X_train_s, y_train)
results["SVR"] = evaluate(svr_model, X_train_s, y_train, X_test_s, y_test, "SVR")

pd.DataFrame(results).T.drop(columns="model").round(3)

**结果解析**：
- 未做标准化时，`kNN (k=10, raw)` 的 Test RMSE 约 `0.955`；标准化后 `kNN (k=10, scaled)` 的 Test RMSE 降至 `0.733`，误差减少了超过 20%。
- 在标准化特征上，`SVR`（默认 RBF 核）的 Test RMSE 进一步降到 `0.695`，$R^2 = 0.898$，略优于标准化后的 kNN。
- **注意**：如果把未标准化的 `X_train` 直接喂给 `SVR()`，RBF 核函数 $\exp(-\gamma \|x - x'\|^2)$ 会完全被数值在几百量级的 `MolWt` 支配，Test RMSE 会退化到 `1.13` 左右。

### 练习 02a-3：组合“描述符 + 计数指纹”训练随机森林

**题目回顾**：把 10 个物理化学描述符与 2048 维 **计数指纹（Count Morgan FP）** 拼接起来（共 2058 维），训练 `RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)`，存入 `results["RF + desc + count FP"]`。

In [ ]:
# 1. 生成 2048 维 Bit 指纹和 Count 指纹
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X_fp_bit   = np.vstack([fp_gen.GetFingerprintAsNumPy(m) for m in df["mol"]])
X_fp_count = np.vstack([fp_gen.GetCountFingerprintAsNumPy(m) for m in df["mol"]])

fp_train, fp_test   = X_fp_bit[idx_train], X_fp_bit[idx_test]
fpc_train, fpc_test = X_fp_count[idx_train], X_fp_count[idx_test]

# 先补齐 02a 正文里的几个基准模型方便横向对比
rf_desc = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)
results["RF (10 descriptors)"] = evaluate(rf_desc, X_train, y_train, X_test, y_test, "RF (10 descriptors)")

rf_bit = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1).fit(fp_train, y_train)
results["RF + bit FP"] = evaluate(rf_bit, fp_train, y_train, fp_test, y_test, "RF + bit FP")

rf_count = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1).fit(fpc_train, y_train)
results["RF + count FP"] = evaluate(rf_count, fpc_train, y_train, fpc_test, y_test, "RF + count FP")

X_comb_tr = np.hstack([X_train.values, fp_train])
X_comb_te = np.hstack([X_test.values,  fp_test])
rf_comb = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1).fit(X_comb_tr, y_train)
results["RF + desc + bit FP"] = evaluate(rf_comb, X_comb_tr, y_train, X_comb_te, y_test, "RF + desc + bit FP")

# 2. 练习 02a-3 解答：拼接 10 个描述符 + 2048 维 Count 指纹
X_combc_tr = np.hstack([X_train.values, fpc_train])
X_combc_te = np.hstack([X_test.values,  fpc_test])

rf_combc = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_combc.fit(X_combc_tr, y_train)
results["RF + desc + count FP"] = evaluate(
    rf_combc, X_combc_tr, y_train, X_combc_te, y_test, "RF + desc + count FP"
)

pd.DataFrame(results).T.drop(columns="model").sort_values("Test RMSE").round(3)

**结果解析**：
- `RF + desc + count FP` 的 Test RMSE 约为 `0.669`，略好于 `RF + desc + bit FP`（`0.674`），两者都显著优于单独使用指纹（`1.093` / `0.946`）或单独使用 10 个描述符（`0.688`）。
- **为什么“描述符 + Bit 指纹”和“描述符 + Count 指纹”差距变小了？**
  单独用 Bit 指纹时，最大的短板是丢失了“分子大小 / 重复基团数量”（比如庚烷和十六烷的 Bit 指纹完全一样）；但当我们把 `MolWt`、`HeavyAtomCount`、`MolLogP` 等全局描述符拼接进去后，分子尺寸信息已经由这 10 个描述符补齐了，因此 Bit 指纹的短板被很大程度弥补了。

---

# 02b — 模型验证、超参数调优与适用域（Validation, Tuning & AD）练习答案

### 练习 02b-1：对比默认随机森林与调参后随机森林的训练/测试误差

**题目回顾**：训练一棵默认参数的 `RandomForestRegressor(random_state=42)`，对比它和 `GridSearchCV` 选出的最优模型 `rf_gs.best_estimator_` 在训练集与测试集上的 RMSE。

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
calc_rmse = lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))

# 先复现 02b 中的 GridSearchCV
param_grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [6, 12, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 1.0],
}
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    n_jobs=-1,
).fit(X_train, y_train)

# 默认随机森林
rf_default = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)

comp_02b1 = pd.DataFrame([
    {
        "Model": "RF (default: n_estimators=100, max_depth=None)",
        "Train RMSE": calc_rmse(y_train, rf_default.predict(X_train)),
        "Test RMSE":  calc_rmse(y_test,  rf_default.predict(X_test)),
    },
    {
        "Model": f"RF (GridSearchCV best: {rf_gs.best_params_})",
        "Train RMSE": calc_rmse(y_train, rf_gs.predict(X_train)),
        "Test RMSE":  calc_rmse(y_test,  rf_gs.predict(X_test)),
    },
]).set_index("Model")
comp_02b1.round(4)

**结果解析**：
- 在这 10 个与溶解度高度相关的物理化学描述符上，默认随机森林（`max_depth=None, min_samples_leaf=1, max_features=1.0`）本身就已经接近最优，调参后（`n_estimators=300, max_depth=12`）Test RMSE 仅从 `0.688` 微降至 `0.685`。
- 这正是随机森林在材料与化学表格数据上广受欢迎的原因：**对默认超参数非常稳健（开箱即用）**。相比之下，下面练习 02b-2 中的 kNN 或 SVR 对超参数就敏感得多。

### 练习 02b-2：用 Pipeline + GridSearchCV 为 kNN 调优

**题目回顾**：给 `make_pipeline(StandardScaler(), KNeighborsRegressor())` 做网格搜索，搜索空间：
- `kneighborsregressor__n_neighbors`: `[3, 5, 10, 20, 40]`
- `kneighborsregressor__weights`: `["uniform", "distance"]`

In [ ]:
knn_pipe = make_pipeline(StandardScaler(), KNeighborsRegressor())

param_grid_knn = {
    "kneighborsregressor__n_neighbors": [3, 5, 10, 20, 40],
    "kneighborsregressor__weights": ["uniform", "distance"],
}

knn_gs = GridSearchCV(
    knn_pipe,
    param_grid=param_grid_knn,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    n_jobs=-1,
)
knn_gs.fit(X_train, y_train)

print("最优超参数:", knn_gs.best_params_)
print(f"5-Fold CV 最优 RMSE : {-knn_gs.best_score_:.3f}")
print(f"独立测试集 Test RMSE: {calc_rmse(y_test, knn_gs.predict(X_test)):.3f}")

# 把全部 10 组参数组合的 CV 结果整理成透视表
cv_table = pd.DataFrame(knn_gs.cv_results_)
cv_table["CV_RMSE"] = -cv_table["mean_test_score"]
cv_table.pivot(
    index="param_kneighborsregressor__n_neighbors",
    columns="param_kneighborsregressor__weights",
    values="CV_RMSE",
).round(3)

**结果解析**：
- 最优组合是 `n_neighbors=5, weights="distance"`（按距离倒数加权，离得越近的邻居话语权越大），测试集 RMSE 从默认 `uniform, k=10` 的 `0.733` 显著降到了 **`0.658`**，甚至超过了随机森林在 10 个描述符上的表现（`0.685`）！
- 从透视表还能看到：不论 `k` 取多少，`weights="distance"` 一律优于等权重的 `"uniform"`。原因在于化学空间中即使挑出最近的 10 个分子，第 1 近邻（可能是同系物）和第 10 近邻的相似度差异也很大，按距离加权能避免远邻的噪声干扰。

### 练习 02b-3：用 Count 指纹重新计算并五苯（pentacene）的最近邻相似度

**题目回顾**：在 `02b` 正文中，用 0/1 Bit 指纹算出的并五苯与训练集最大 Tanimoto 相似度高达 `1.000`（因为并五苯和蒽、并四苯的局部环碎片种类完全相同）。请改用 `fp_gen.GetCountFingerprint(mol)`（计数指纹）重新计算并五苯与训练集的最大 Tanimoto 相似度，以及 Top-5 最近邻。

In [ ]:
train_mols = df["mol"].iloc[idx_train].tolist()
train_names = df["name"].iloc[idx_train].tolist()

train_bit_fps   = [fp_gen.GetFingerprint(m) for m in train_mols]
train_count_fps = [fp_gen.GetCountFingerprint(m) for m in train_mols]

pentacene_mol = Chem.MolFromSmiles("c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1")
q_bit   = fp_gen.GetFingerprint(pentacene_mol)
q_count = fp_gen.GetCountFingerprint(pentacene_mol)

sims_bit   = DataStructs.BulkTanimotoSimilarity(q_bit, train_bit_fps)
sims_count = DataStructs.BulkTanimotoSimilarity(q_count, train_count_fps)

print(f"Bit   FP 下 pentacene 与训练集最大相似度: {max(sims_bit):.3f}")
print(f"Count FP 下 pentacene 与训练集最大相似度: {max(sims_count):.3f}\n")

top5_idx = np.argsort(sims_count)[::-1][:5]
pd.DataFrame({
    "Train Molecule": [train_names[i] for i in top5_idx],
    "Count FP Tanimoto": [round(sims_count[i], 3) for i in top5_idx],
    "Bit FP Tanimoto":   [round(sims_bit[i], 3) for i in top5_idx],
})

**结果解析**：
- 在 `Bit FP` 下，并五苯与训练集中并四苯（tetracene）、蒽（anthracene）的 Tanimoto 相似度都是 **`1.000`**（完全撞车，无法区分 3 个环、4 个环还是 5 个环）。
- 换成 `Count FP` 后，由于每个芳香碳碎片出现的次数不同，并五苯与并四苯的相似度降为 **`0.822`**，与蒽降为 **`0.630`**，与萘降为 **`0.415`**——成功按稠环尺寸拉开了梯度！这也再次印证了处理同系物或稠环体系时，**Count 指纹比 Bit 指纹保留了关键的化学计量与尺寸信息**。

---

# 02c — 分类与基础特征重要性（Classification & Feature Importance）练习答案

先运行 `02c` 中的分类标签定义与分层抽样划分：

In [ ]:
THRESHOLD = -4.0
y_cls = (df["logS"] < THRESHOLD).astype(int)   # 1 = insoluble (logS < -4)

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X, y_cls, test_size=0.2, stratify=y_cls, random_state=RANDOM_STATE
)

def eval_classifier(clf, X_te, y_te):
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    return {
        "Accuracy":  accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall":    recall_score(y_te, y_pred, zero_division=0),
        "F1":        f1_score(y_te, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_te, y_prob),
    }

### 练习 02c-1：加入 kNN 与梯度提升分类器对比

**题目回顾**：在 `models` 字典中加入 `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10))` 和 `GradientBoostingClassifier(random_state=42)`，对比全部五个模型在测试集上的五项分类指标。

In [ ]:
models = {
    "Dummy (most_frequent)": DummyClassifier(strategy="most_frequent").fit(X_tr_c, y_tr_c),
    "LogisticRegression":    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr_c, y_tr_c),
    "kNN (k=10, scaled)":    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=10)).fit(X_tr_c, y_tr_c),
    "GradientBoosting":      GradientBoostingClassifier(random_state=RANDOM_STATE).fit(X_tr_c, y_tr_c),
    "RandomForest":          RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1).fit(X_tr_c, y_tr_c),
}

pd.DataFrame({k: eval_classifier(m, X_te_c, y_te_c) for k, m in models.items()}).T.round(3)

**结果解析**：
- 四个真实模型的 ROC-AUC 都在 `0.946 ~ 0.955` 之间，整体排序能力都很强。
- 其中 `RandomForest` 和 `GradientBoosting` 的综合 F1 最高（约 `0.81 ~ 0.83`）；`kNN (k=10)` 的 Precision 较高（`0.855`），意味着它报出的难溶分子误报率低，但 Recall 略低（`0.734`）。

### 练习 02c-2：极端难溶分子筛查（严重类别不平衡）

**题目回顾**：将难溶阈值改为 `-6.0`（`y_extreme = (df["logS"] < -6.0).astype(int)`），观察正类占比，并用 `stratify=y_extreme` 重新划分训练/测试集，对比 `DummyClassifier` 与 `RandomForestClassifier` 的 `Accuracy` 和 `F1`。

In [ ]:
y_extreme = (df["logS"] < -6.0).astype(int)
print(f"logS < -6 的极端难溶分子占比: {y_extreme.mean():.2%} ({y_extreme.sum()} / {len(y_extreme)})\n")

X_tr_ex, X_te_ex, y_tr_ex, y_te_ex = train_test_split(
    X, y_extreme, test_size=0.2, stratify=y_extreme, random_state=RANDOM_STATE
)

dummy_ex = DummyClassifier(strategy="most_frequent").fit(X_tr_ex, y_tr_ex)
rf_ex    = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1).fit(X_tr_ex, y_tr_ex)

pd.DataFrame({
    "Dummy (永远猜 0)": eval_classifier(dummy_ex, X_te_ex, y_te_ex),
    "RandomForest":     eval_classifier(rf_ex,    X_te_ex, y_te_ex),
}).T.round(3)

**结果解析**：
- 当正类占比降到 `11.1%` 时，什么都不学的 `DummyClassifier` 光靠全猜 `0` 就能拿到 **`88.9%` 的 Accuracy**！如果论文或报告里只写“模型准确率达到 89%”，看似很高，实则毫无意义。
- 看 `F1` 和 `ROC-AUC` 才能一眼区分真伪：Dummy 的 F1 是 `0.000`，而 `RandomForest` 的 F1 达到 `0.816`（Precision `0.833`，Recall `0.800`，ROC-AUC `0.979`）。

### 练习 02c-3：揪出模型“漏报”最严重的难溶分子并分析原因

**题目回顾**：在 `THRESHOLD = -4.0` 的测试集中，找出真实标签为难溶（`y_te_c == 1`），但随机森林预测的 `P(insoluble)` 最低的前 6 个分子（最严重的假阴性 False Negatives），打印其信息并画出分子结构。

In [ ]:
rf_clf = models["RandomForest"]

test_df = df.loc[X_te_c.index, ["name", "smiles", "logS", "mol"]].copy()
test_df["MolLogP"] = X_te_c["MolLogP"].values
test_df["MolWt"]   = X_te_c["MolWt"].values
test_df["TPSA"]    = X_te_c["TPSA"].values
test_df["True_y"]  = y_te_c.values
test_df["P(insoluble)"] = rf_clf.predict_proba(X_te_c)[:, 1]

# 筛选真实难溶 (True_y == 1)，按预测难溶概率从小到大排序，取前 6 个
fn_worst6 = test_df[test_df["True_y"] == 1].sort_values("P(insoluble)").head(6)
display(fn_worst6[["name", "smiles", "logS", "MolLogP", "MolWt", "TPSA", "P(insoluble)"]].round(3))

legends = [
    f"{row['name'][:18]}\nlogS={row['logS']:.2f}, P={row['P(insoluble)']:.2f}\nLogP={row['MolLogP']:.2f}"
    for _, row in fn_worst6.iterrows()
]
Draw.MolsToGridImage(
    fn_worst6["mol"].tolist(),
    molsPerRow=3,
    subImgSize=(300, 220),
    legends=legends,
)

**深度化学原因分析（为什么模型会漏报这 6 个分子？）**

仔细观察这 6 个被模型误判为“可溶”的难溶分子，可以归结为三类典型的物理化学与数据边界机制：

1. **高对称性 + 强分子间氢键网络导致的“砖头晶格（Brick-dust）”分子**：
   - 典型代表：**对苯二甲酸（terephthalic acid，PET 塑料单体）**。它的分子量很小（`166`），带着两个亲水的羧基，`MolLogP` 只有 `1.18`，`TPSA` 高达 `74.6`。仅看单分子的疏水性（`MolLogP`），模型有 96.5% 的把握认为它极易溶于水（`P(insoluble) = 0.035`）。
   - 但实际上，对苯二甲酸具有高度轴对称的刚性对位结构，在固态下通过羧基二聚体氢键和 $\pi\text{-}\pi$ 堆积形成极其稳固的晶格，**熔点高达 $300\ ^\circ\text{C}$ 以上（升华）**！正如正文 6.3 节 Yalkowsky 方程 $\log S \approx 0.5 - \log P - 0.01(T_m - 25)$ 所示，2D 描述符完全不知道它有如此高的晶格能（熔点项 $T_m$），因而严重高估了它的溶解度。
2. **含金属配位/有机盐的多组分体系**：
   - 典型代表：**`bis(2-ethylhexyl)calcium`（SMILES 中含有 `.` 分隔的钙离子 `[Ca++]` 与有机阴离子）**。RDKit 的常规经验规则在计算含金属离子的多组分盐类 `MolLogP` 时存在系统性偏差，且离子晶体/配合物的溶解热力学与中性有机小分子截然不同。
3. **硬阈值切分带来的边界样本（Borderline Cases）**：
   - 如 `quinomethionate`（$\log S = -4.09$）、`famphur`（$\log S = -4.005$）、`ethion`（$\log S = -4.33$）。它们的真实溶解度本来就紧贴在人为设定的分界线 $-4.0$ 边缘（甚至 $-4.005$ 离“可溶”只差 $0.005$ 个对数单位，完全在实验测量误差范围内）。这也说明：**把连续物理量硬切成二分类时，分界线附近的样本天然存在标签噪声**，若实验关心边界附近的具体数值，应当回到 `02a` 的回归模型。